# 01 — Setup & data infrastructure

Verifies the local environment before the WPW pipeline: data paths (PTB-XL, Ningbo, PTB-XL+), library conformity against `requirements.txt`, and canonical loading of one ECG from each dataset.

**Inputs:** raw data under `data/raw/`. **Outputs:** none (checks only).

> Migration history (Colab to local, steps 0-1 recap): see `JOURNAL.md` sections 9bis/13.

### Block 1.0: Project paths

In [ ]:
import os

ROOT = r'.'
PTBXL_BASE  = os.path.join(ROOT, 'data', 'raw', 'ptbxl',
                           'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3')
NINGBO_ROOT = os.path.join(ROOT, 'data', 'raw', 'ningbo',
                           'a-large-scale-12-lead-electrocardiogram-database-for-arrhythmia-study-1.0.0')
PROCESSED   = os.path.join(ROOT, 'data', 'processed')
PTBXL_PLUS  = os.path.join(ROOT, 'data', 'raw', 'ptbxl_plus')

print("=== Data folders existence ===")
for name, p in [('PTBXL_BASE',PTBXL_BASE),('NINGBO_ROOT',NINGBO_ROOT),
                ('PROCESSED',PROCESSED),('PTBXL_PLUS',PTBXL_PLUS)]:
    print(f"  {'OK     ' if os.path.exists(p) else 'MISSING'} {name}")

assert os.path.exists(os.path.join(PTBXL_BASE,'records500')), 'PTB-XL records500 not found'
assert os.path.exists(os.path.join(NINGBO_ROOT,'WFDBRecords')), 'Ningbo WFDBRecords not found'

### Block 1.1: Environment conformity (requirements.txt)

In [ ]:
# requirements.txt is the single source of truth (manually pinned). This block VERIFIES that
# the installed environment matches it; it does NOT regenerate anything.
import importlib

PIP_TO_IMPORT = {'PyWavelets': 'pywt', 'scikit-learn': 'sklearn'}  # pip name != import name

with open(os.path.join(ROOT, 'requirements.txt')) as f:
    pins = [l.strip() for l in f if l.strip() and not l.startswith('#')]

print("=== Conformity vs requirements.txt ===")
mismatch = []
for pin in pins:
    name, _, want = pin.partition('==')
    mod = PIP_TO_IMPORT.get(name, name)
    try:
        got = getattr(importlib.import_module(mod), '__version__', '?')
    except ImportError:
        got = 'NOT INSTALLED'
    ok = (got == want)
    if not ok: mismatch.append(name)
    print(f"  {'OK ' if ok else '!! '} {name:14s} required {want:14s} | installed {got}")

if mismatch:
    print(f"\n[WARNING] {len(mismatch)} mismatch(es): {mismatch} -> run `pip install -r requirements.txt`")
else:
    print("\n[OK] Environment matches requirements.txt.")

### Block 1.2: Load one PTB-XL ECG

In [ ]:
import wfdb, numpy as np, matplotlib.pyplot as plt

rec = os.path.join(PTBXL_BASE, 'records500', '00000', '00001_hr')
signal, meta = wfdb.rdsamp(rec)
fs = int(meta['fs'])

assert signal.shape == (5000, 12), f"shape {signal.shape} != (5000,12)"
assert fs == 500, f"fs {fs} != 500 Hz"
assert np.isnan(signal).sum() == 0, 'NaN in PTB-XL signal'
print(f"PTB-XL 00001_hr: {signal.shape}, {fs} Hz, {meta['sig_name']}, 0 NaN -- OK")

plt.figure(figsize=(14,3))
plt.plot(np.arange(3*fs)/fs, signal[:3*fs, 1], lw=0.8)
plt.title('PTB-XL 00001_hr -- lead II (3 s)'); plt.xlabel('Time (s)'); plt.ylabel('mV')
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

### Block 1.3: Load one Ningbo ECG (consistency with PTB-XL)

In [ ]:
# wfdb for both datasets (handles .mat transposition + ADC gain -> unified format).
rec = os.path.join(NINGBO_ROOT, 'WFDBRecords/01/010/JS00001')
sig, meta = wfdb.rdsamp(rec)
fs = int(meta['fs'])

assert sig.shape == (5000, 12), f"shape {sig.shape} != (5000,12)"
assert fs == 500, f"fs {fs} != 500 Hz"
assert np.isnan(sig).sum() == 0, 'NaN in Ningbo signal'
print(f"Ningbo JS00001: {sig.shape}, {fs} Hz, units {meta['units'][1]}, 0 NaN -- OK")
print(f"Max |lead II| amplitude: {np.abs(sig[:,1]).max():.2f} mV (consistent with PTB-XL)")

plt.figure(figsize=(14,3))
plt.plot(np.arange(3*fs)/fs, sig[:3*fs, 1], lw=0.8, color='#ff7f0e')
plt.title('Ningbo JS00001 -- lead II (3 s)'); plt.xlabel('Time (s)'); plt.ylabel('mV')
plt.grid(alpha=.3); plt.tight_layout(); plt.show()